In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

In [47]:
import pdb

<p style='font-size:20px'>Classification</p>

In [8]:
df = pd.read_csv('iris.csv')
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [10]:
from sklearn.preprocessing import LabelEncoder

labels = ['setosa', 'versicolor', 'virginica']
label_encoder = LabelEncoder()

In [12]:
X = df.drop('species', axis=1)
y = label_encoder.fit_transform(df['species'])

y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

scaler = StandardScaler()
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)

dataset_train = TensorDataset(torch.tensor(scaled_X_train,  dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
dataset_test = TensorDataset(torch.tensor(scaled_X_test,  dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

In [16]:
dataloader_train = DataLoader(dataset_train, batch_size=5, shuffle=True)

counter = 0

for batch_inputs, batch_labels in dataloader_train:
    if counter == 0:
        print('batch inputs shape:', batch_inputs.shape)
        print('batch inputs datatype:', batch_inputs.dtype)
        print('batch labels shape:', batch_labels.shape)
        print('batch labels datatype:', batch_labels.dtype)
    
    counter += 1
    
    if counter == 10:
        print(batch_inputs)
        print(batch_labels)

batch inputs shape: torch.Size([5, 4])
batch inputs datatype: torch.float32
batch labels shape: torch.Size([5])
batch labels datatype: torch.int64
tensor([[-0.1710, -0.6633,  0.1886,  0.1227],
        [ 2.4447,  1.7057,  1.4570,  0.9981],
        [ 0.8991, -0.1895,  0.3540,  0.2477],
        [-1.2410,  0.0474, -1.1902, -1.2529],
        [-0.8844,  0.7581, -1.2454, -1.2529]])
tensor([1, 2, 1, 0, 0])


In [59]:
model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16,3))
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [61]:
num_epochs = 100

for i in range(num_epochs):
    for features, target in dataloader_train:
        #features = features.float()
        #target = target.float()
        #target = target.unsqueeze(1)

        optimizer.zero_grad()    # otherwise the gradients keep being added up across batches
        
        prediction = model(features) 
        loss = criterion(prediction, target)    
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)    # Prevents exploding gradients by scaling down the gradients so their total norm does not exceed 1.0
        loss.backward()
        optimizer.step()

        #print('Batch done.')

    if (i+1) % 10 == 0:
        print(f'{round((i+1)/num_epochs*100,2)} % done.')

#pdb.set_trace()

10.0 % done.
20.0 % done.
30.0 % done.
40.0 % done.
50.0 % done.
60.0 % done.
70.0 % done.
80.0 % done.
90.0 % done.
100.0 % done.


In [63]:
dataloader_test = DataLoader(dataset_test, batch_size=len(dataset_test), shuffle=False)
model.eval()

correct_predictions = 0
predictions = 0

with torch.no_grad():
  for features, target in dataloader_test:
      
      outputs = model(features)
      _, pred_class = torch.max(outputs, 1)  # Get the predicted class index
      
      correct_predictions += (pred_class == target).sum().item()  # Sum the number of correct predictions
      predictions += target.size(0)  # Count total number of predictions
      
acc = correct_predictions/predictions
print(f'Accuracy: {round(acc, 2)}')

# Set the model back to training mode
model.train()

Accuracy: 0.98


Sequential(
  (0): Linear(in_features=4, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=3, bias=True)
)

In [65]:
correct_predictions

44